<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Regression-based Rating Score Prediction using Embedding Features**


Estimated time needed: **45** minutes


## **Introduction**


In the previous lab (6.3.3), we trained a neural network to predict user–item interactions (ratings).
While training the neural network, the model also automatically learned the user and item embedding features.

An embedding feature is a small group of numbers that represents hidden features of a user or an item. For example: 
- a user embedding features may represent the user's preferences
- an item embedding features may represent the properties of the item. 

### **Idea of this lab**

In this lab, we will use the learned user embeddings and item embeddings as input features for a regression model. The embedding vectors already contain useful information about users and items. If the embeddings are good, a simple regression model should be able to use them to predict the rating.

We will do: user embedding + item embedding → regression model → rating

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/module_4/images/rating_regression.png)


For each user–item pair, we will do the following:
1. get the embedding vector of the user
2. get the embedding vector of the item
3. combine the two embedding vectors into one feature vector
4. use this feature vector as the input `X`
5. use the rating as the output `Y`

This converts the recommendation problem into a normal supervised learning problem. We will try several regression models and compare their performance.

### **Goal of this lab**
Embedding features learned from a neural network can be used as input features for traditional machine learning models.

## **Objectives**


* Build regression models to predict ratings using the combined embedding vectors


----


### **Prepare and setup lab environment**


First install and import required libraries:


In [ ]:
# %pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np 

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [ ]:
# also set a random state
rs = 123

### **Load embedding datasets**


In [ ]:
rating_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-ML0321EN-Coursera/labs/v2/module_3/ratings.csv"
user_emb_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/user_embeddings.csv"
item_emb_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_embeddings.csv"

The first dataset is the rating dataset that contains a user-item interaction matrix


In [ ]:
rating_df = pd.read_csv(rating_url)

In [ ]:
rating_df.head()

As you can see from the above data, the user and item are just ids, let's substitute them by their embedding vectors:


In [ ]:
# Load user embeddings
user_emb = pd.read_csv(user_emb_url)
# Load item embeddings
item_emb = pd.read_csv(item_emb_url)

In [ ]:
user_emb.head()

In [ ]:
item_emb.head()

In [ ]:
# Merge user embedding features
user_emb_merged = pd.merge(rating_df, user_emb, how='left', left_on='user', right_on='user').fillna(0)
# Merge course embedding features
merged_df = pd.merge(user_emb_merged, item_emb, how='left', left_on='item', right_on='item').fillna(0)

In [ ]:
merged_df.head()

### **Create interaction datasets**

Next, we can combine the user features (the column labels starting with `UFeature` and item features (the column labels starting with `CFeature`. *In machine learning, there are many ways to aggregate two feature vectors such as element-wise add, multiply, max/min, average, etc.* Here we simply add the two sets of feature columns:


In [ ]:
# Define column names for user and course embedding features
u_features = [f"UFeature{i}" for i in range(16)] # Assuming there are 16 user embedding features
c_features = [f"CFeature{i}" for i in range(16)]  # Assuming there are 16 course embedding features

# Extract user embedding features
user_embeddings = merged_df[u_features]
# Extract course embedding features
course_embeddings = merged_df[c_features]
# Extract ratings
ratings = merged_df['rating']

# Aggregate the two feature columns using element-wise add
regression_dataset = user_embeddings + course_embeddings.values
# Rename the columns of the resulting DataFrame
regression_dataset.columns = [f"Feature{i}" for i in range(16)]# Assuming there are 16 features
# Add the 'rating' column from the original DataFrame to the regression dataset
regression_dataset['rating'] = ratings
# Display the first few rows of the regression dataset
regression_dataset.head()

By now, we have built the input dataset `X` and the output vector `y`:


In [ ]:
X = regression_dataset.iloc[:, :-1]
y = regression_dataset.iloc[:, -1]
print(f"Input data shape: {X.shape}, Output data shape: {y.shape}")

### **TASK: Perform regression on the interaction dataset**


Now our input data `X` and output `y` are ready, let's build regression models to map X to y and predict ratings. 


You may use `sklearn` to train and evaluate various regression models.


_TODO: First split dataset into training and testing datasets_


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=rs)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

<details>
    <summary>Click here for Hints</summary>
    
Use `train_test_split()` to split dataset into training and testing datasets.  Use `X, y` as input dataset and output vector. Don't forget to specify `random_state = rs` and `test_size=0.3`.


#### **Linear regression**

_TODO: Create a basic linear regression model_


In [ ]:
### WRITE YOUR CODE HERE
linear_regression = LinearRegression()

<details>
    <summary>Click here for Hints</summary>
    
You can call `linear_regression = LinearRegression()` method to create a Linear Regression object


_TODO: Train the basic regression model with training data_


In [ ]:
### WRITE YOUR CODE HERE
linear_regression.fit(X_train, y_train)

<details>
    <summary>Click here for Hints</summary>
    
You can call `model.fit()` method with `X_train, y_train` parameters.


_TODO: Evaluate the basic regression model_


In [ ]:
y_pred_lr = linear_regression.predict(X_test)
### The main evaluation metric is RMSE but you may use other metrics as well
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
print("Linear Regression RMSE:", rmse)

<details>
    <summary>Click here for Hints</summary>
    
You can call `model.predict()` method with `X_test` parameter to get model predictions. Then use `mean_squared_error()` with `y_test, your_predictions` parameters to calculate the RMSE. 


_TODO: Try different regression models such as Ridge, Lasso, ElasticNet and tune their hyperparameters to see which one has the best performance_


#### **Regularization regression**

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

models = {
    "Ridge_0.1": Ridge(alpha=0.1),
    "Ridge_1.0": Ridge(alpha=1.0),
    "Ridge_10.0": Ridge(alpha=10.0),
    "Lasso_0.001": Lasso(alpha=0.001, max_iter=10000),
    "Lasso_0.01": Lasso(alpha=0.01, max_iter=10000),
    "Lasso_0.1": Lasso(alpha=0.1, max_iter=10000),
    "ElasticNet_0.001": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000),
    "ElasticNet_0.01": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
    "ElasticNet_0.1": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results.append((name, rmse))

results_df = pd.DataFrame(results, columns=["Model", "RMSE"]).sort_values("RMSE")
results_df

## **Summary**


In this lab, we have built regression models to predict numerical course ratings using the embedding feature vectors extracted from neural networks. In the next lab, we can treat the prediction problem as a classification problem as rating only has two categorical values so classification can be a more natural problem statement.


## **Authors**


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/), Su Wu


Copyright © 2021 IBM Corporation. All rights reserved.
